# 多模态共享LoRA梯度冲突分析 Notebook

本Notebook用于对你当前的梯度日志（`gradient_logs.jsonl`）进行**机制研究导向**的数据分析，重点支持：

1. `all / text_only / image_only` 三类分区梯度对比  
2. 层级（layer-depth）与参数类型（adapter_type）的冲突与主导关系分析  
3. `grad_was_none` 未连通梯度诊断  
4. 为论文第3章生成可直接使用的统计图与结论素材  

> 当前日志是统计量（norm/mean/std），若你要做“真实梯度方向余弦+子空间SVD”，需额外保存梯度向量（见最后的可选扩展部分）。


In [ ]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
pd.set_option('display.max_columns', 200)


In [ ]:
# ===== 配置区 =====
LOG_PATH = Path('gradient_logs.jsonl')   # 改成你的真实路径
OUTPUT_DIR = Path('analysis_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert LOG_PATH.exists(), f'文件不存在: {LOG_PATH.resolve()}'
print('Using log file:', LOG_PATH.resolve())


In [ ]:
def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f'[WARN] line {ln} parse failed: {e}')
    return pd.DataFrame(rows)


df = load_jsonl(LOG_PATH)
print('rows =', len(df), 'cols =', len(df.columns))
df.head(3)


In [ ]:
# 兼容 modality_ids 可能为字符串的情况
if 'modality_ids' in df.columns:
    def parse_modality_ids(x):
        if isinstance(x, list):
            return x
        if pd.isna(x):
            return []
        if isinstance(x, str):
            x = x.strip()
            if not x:
                return []
            try:
                y = json.loads(x)
                return y if isinstance(y, list) else []
            except Exception:
                try:
                    y = ast.literal_eval(x)
                    return y if isinstance(y, list) else []
                except Exception:
                    return []
        return []
    df['modality_ids'] = df['modality_ids'].apply(parse_modality_ids)
else:
    df['modality_ids'] = [[] for _ in range(len(df))]

# 衍生字段
required_numeric = ['step', 'layer_depth', 'grad_norm', 'grad_mean', 'grad_std', 'grad_abs_mean', 'numel', 'supervised_token_count']
for c in required_numeric:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

if 'grad_was_none' in df.columns:
    df['grad_was_none'] = df['grad_was_none'].fillna(False).astype(bool)
else:
    df['grad_was_none'] = False

if 'grad_partition' not in df.columns:
    df['grad_partition'] = 'all'

# 批次模态统计

def ratio_image(mods):
    # 1=image_text, 2=video_text 视为含视觉模态
    if not mods:
        return np.nan
    arr = np.array(mods)
    return float(np.mean(np.isin(arr, [1, 2])))

def ratio_text_only(mods):
    if not mods:
        return np.nan
    arr = np.array(mods)
    return float(np.mean(arr == 0))

df['image_batch_ratio'] = df['modality_ids'].apply(ratio_image)
df['text_only_batch_ratio'] = df['modality_ids'].apply(ratio_text_only)

df[['step','grad_partition','layer_depth','adapter_type','grad_norm','supervised_token_count','grad_was_none','image_batch_ratio']].head(5)


## 一、数据质量与覆盖度检查

这部分用于判断日志是否足够支持论文结论（例如分区是否齐全、层数是否覆盖、None梯度比例是否可控）。

In [ ]:
summary = {
    'n_rows': len(df),
    'step_min': int(df['step'].min()) if 'step' in df else None,
    'step_max': int(df['step'].max()) if 'step' in df else None,
    'n_unique_params': int(df['param_name'].nunique()) if 'param_name' in df else None,
    'n_layers': int(df['layer_depth'].nunique()) if 'layer_depth' in df else None,
    'partitions': sorted(df['grad_partition'].dropna().unique().tolist()),
    'none_grad_ratio': float(df['grad_was_none'].mean()) if 'grad_was_none' in df else None,
}
summary


In [ ]:
plt.figure(figsize=(8,4))
order = ['all', 'text_only', 'image_only']
ax = sns.countplot(data=df, x='grad_partition', order=[p for p in order if p in df['grad_partition'].unique()])
ax.set_title('Partition Record Count')
ax.set_xlabel('grad_partition')
ax.set_ylabel('count')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_partition_count.png', dpi=180)
plt.show()


## 二、核心可视化1：层级-分区梯度强度热力图

用于回答：**哪一层在 text/image 分区上冲突更明显？**

In [ ]:
layer_partition = (
    df.groupby(['layer_depth', 'grad_partition'], as_index=False)['grad_norm']
      .mean()
)
pivot_lp = layer_partition.pivot(index='layer_depth', columns='grad_partition', values='grad_norm').sort_index()

plt.figure(figsize=(10, 8))
sns.heatmap(pivot_lp, cmap='magma', cbar_kws={'label':'mean grad_norm'})
plt.title('Layer vs Partition Mean Grad Norm')
plt.xlabel('grad_partition')
plt.ylabel('layer_depth')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_layer_partition_gradnorm_heatmap.png', dpi=200)
plt.show()

pivot_lp.tail()


## 三、核心可视化2：训练过程趋势（step维度）

用于回答：冲突在训练过程中是增强、减弱还是震荡。

In [ ]:
trend = (
    df.groupby(['step', 'grad_partition'], as_index=False)['grad_norm']
      .mean()
)
plt.figure(figsize=(12,5))
sns.lineplot(data=trend, x='step', y='grad_norm', hue='grad_partition', linewidth=2)
plt.title('Mean Grad Norm over Steps by Partition')
plt.xlabel('step')
plt.ylabel('mean grad_norm')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_step_partition_gradnorm_trend.png', dpi=180)
plt.show()


## 四、核心可视化3：模态主导性（text vs image）

### 指标定义（论文可直接写）
对于同一步、同一参数，定义主导指数：

\[
D = rac{\|g_{image}\| - \|g_{text}\|}{\|g_{image}\| + \|g_{text}\| + \epsilon}
\]

- \(D>0\)：视觉分区主导
- \(D<0\)：文本分区主导
- \(|D|\) 越大说明竞争越激烈


In [ ]:
eps = 1e-12
sub = df[df['grad_partition'].isin(['text_only', 'image_only'])].copy()

key_cols = ['step', 'param_name', 'layer_depth', 'adapter_type']
wide = sub.pivot_table(index=key_cols, columns='grad_partition', values='grad_norm', aggfunc='mean').reset_index()
wide = wide.dropna(subset=['text_only', 'image_only'])
wide['dominance_index'] = (wide['image_only'] - wide['text_only']) / (wide['image_only'] + wide['text_only'] + eps)

plt.figure(figsize=(11,5))
sns.histplot(wide['dominance_index'], bins=60, kde=True)
plt.axvline(0, color='red', linestyle='--', linewidth=1.5)
plt.title('Dominance Index Distribution (image vs text)')
plt.xlabel('dominance_index')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_dominance_hist.png', dpi=180)
plt.show()

wide[['dominance_index']].describe()


In [ ]:
dom_layer = wide.groupby('layer_depth', as_index=False)['dominance_index'].mean().sort_values('layer_depth')
plt.figure(figsize=(11,5))
sns.lineplot(data=dom_layer, x='layer_depth', y='dominance_index', marker='o')
plt.axhline(0, color='red', linestyle='--', linewidth=1.2)
plt.title('Layer-wise Mean Dominance Index')
plt.xlabel('layer_depth')
plt.ylabel('mean dominance_index')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_layer_dominance.png', dpi=180)
plt.show()


## 五、核心可视化4：`grad_was_none` 机制诊断

用于判断“未连通梯度”是否集中在某些层/模块/分区。

In [ ]:
none_stat = (
    df.groupby(['layer_depth', 'grad_partition'], as_index=False)['grad_was_none']
      .mean()
)
pivot_none = none_stat.pivot(index='layer_depth', columns='grad_partition', values='grad_was_none').sort_index()

plt.figure(figsize=(10,8))
sns.heatmap(pivot_none, cmap='Blues', vmin=0, vmax=max(0.01, np.nanmax(pivot_none.values)), cbar_kws={'label':'none ratio'})
plt.title('Layer vs Partition None-Gradient Ratio')
plt.xlabel('grad_partition')
plt.ylabel('layer_depth')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'fig_layer_partition_none_ratio.png', dpi=200)
plt.show()

none_stat.head()


## 六、核心可视化5：All分区与子分区的一致性检查

如果 `all` 与 `text_only/image_only` 差异异常，可提示标签mask、分区定义或梯度路径设置问题。

In [ ]:
all_df = df[df['grad_partition'] == 'all'][['step','param_name','layer_depth','grad_norm']].rename(columns={'grad_norm':'all_norm'})
text_df = df[df['grad_partition'] == 'text_only'][['step','param_name','layer_depth','grad_norm']].rename(columns={'grad_norm':'text_norm'})
img_df = df[df['grad_partition'] == 'image_only'][['step','param_name','layer_depth','grad_norm']].rename(columns={'grad_norm':'image_norm'})

chk = all_df.merge(text_df, on=['step','param_name','layer_depth'], how='inner')           .merge(img_df, on=['step','param_name','layer_depth'], how='inner')
if len(chk) > 0:
    chk['sub_sum'] = chk['text_norm'] + chk['image_norm']
    chk['all_minus_subsum'] = chk['all_norm'] - chk['sub_sum']

    plt.figure(figsize=(11,5))
    sns.histplot(chk['all_minus_subsum'], bins=60, kde=True)
    plt.axvline(0, color='red', linestyle='--')
    plt.title('all_norm - (text_norm + image_norm)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fig_all_vs_subsum_diff.png', dpi=180)
    plt.show()

    display(chk[['all_norm','text_norm','image_norm','sub_sum','all_minus_subsum']].describe())
else:
    print('No overlap records among all/text_only/image_only for consistency check.')


## 七、可直接写入论文的统计表导出

导出后你可以直接做表3-1/3-2/3-3。

In [ ]:
table_layer_partition = (
    df.groupby(['layer_depth', 'grad_partition'])
      .agg(mean_grad_norm=('grad_norm','mean'),
           std_grad_norm=('grad_norm','std'),
           mean_none_ratio=('grad_was_none','mean'),
           mean_token_count=('supervised_token_count','mean'))
      .reset_index()
)

TABLE_PATH = OUTPUT_DIR / 'table_layer_partition_stats.csv'
table_layer_partition.to_csv(TABLE_PATH, index=False, encoding='utf-8-sig')
print('Saved:', TABLE_PATH.resolve())
table_layer_partition.head(10)


## 八、可选扩展：真实方向冲突（余弦）与子空间容量（SVD）

> 你现在日志只有统计量，无法恢复完整梯度向量，因此**不能精确计算余弦相似度与SVD容量**。  
> 如果你后续在日志里增加 `grad_vector_path`（每条记录对应 `.npy` 向量），即可执行下方代码。


In [ ]:
# 可选：当日志包含 grad_vector_path 时启用
if 'grad_vector_path' in df.columns:
    import numpy.linalg as LA

    sample = df.dropna(subset=['grad_vector_path']).head(200).copy()

    def load_vec(p):
        p = Path(p)
        return np.load(p)

    # 示例：同step同param下 text/image 余弦
    subset = sample[sample['grad_partition'].isin(['text_only','image_only'])]
    key = ['step','param_name','layer_depth']

    txt = subset[subset['grad_partition']=='text_only'][key+['grad_vector_path']].rename(columns={'grad_vector_path':'text_path'})
    img = subset[subset['grad_partition']=='image_only'][key+['grad_vector_path']].rename(columns={'grad_vector_path':'img_path'})
    pair = txt.merge(img, on=key, how='inner')

    cos_vals = []
    for _, r in pair.iterrows():
        vt = load_vec(r['text_path']).reshape(-1)
        vi = load_vec(r['img_path']).reshape(-1)
        denom = (LA.norm(vt)*LA.norm(vi) + 1e-12)
        cos_vals.append(float(np.dot(vt, vi)/denom))

    pair['cosine'] = cos_vals
    display(pair['cosine'].describe())

    plt.figure(figsize=(8,4))
    sns.histplot(pair['cosine'], bins=40, kde=True)
    plt.title('Cosine(text_grad, image_grad)')
    plt.show()
else:
    print('当前日志未包含 grad_vector_path，跳过余弦/SVD精确分析。')


## 九、结论模板（写作提示）

你可以按以下模板组织第3章结果：

1. **异质性证据**：`text_only` 与 `image_only` 在多层的 `grad_norm` 分布显著不同。  
2. **竞争证据**：主导指数在0两侧分布，说明不同层/参数在不同模态间存在更新竞争。  
3. **瓶颈证据**：若某些层长期 `|dominance_index|` 高且 `all` 分区收益不稳定，说明共享子空间承载冲突。  
4. **机制证据**：`grad_was_none` 聚集层可视为当前训练图对该分区“不可达/弱可达”的结构信号。  
5. **方法动机**：自然引出第4章“模态分离或容量重分配”方案。
